## Step1: dataset loading

In [1]:
!PYTHONPATH=../../src python3 ../../src/datasets/main.py dataset=imagenet forget=instance dataset.init_dir="imagenet_example_data" dataset.save_dir="imagenet_example_split" dataset.val_ratio=0.1 forget.forget_idx=[0]


============ Run Configuration ============
seed: 42
dataset:
  name: imagenet
  init_dir: imagenet_example_data
  save_dir: imagenet_example_split
  proportion: 1
  val_ratio: 0.1
forget:
  method: instance
  forget_idx:
  - 0
  retain_size: null

Renamed dataset created at: imagenet_example_data_renamed
save dir imagenet_example_split


## Step2: model loading

In [2]:
!PYTHONPATH=../../src python3 -m train.main dataset=imagenet model=resnet18 wandb_cfg=default train_cfg=pretrained dataset.load_dir="imagenet_example_split" model.pretrained=True model.num_classes=1000 model.save_dir="artifacts/models"


============ Run Configuration ============
seed: 42
model:
  name: resnet18
  pretrained: true
  freeze_all_except_classifier: false
  num_classes: 1000
  checkpoint_path: null
  save_dir: artifacts/models
dataset:
  name: imagenet
  load_dir: imagenet_example_split
  batch_sizes:
    train: 64
    val: 64
  num_workers: 4
train_cfg:
  epochs: 0
  lr: 0
  weight_decay: 0
wandb_cfg:
  project_name: machine_unlearning_001
  run_id: train_job_001

Model downloaded without training at artifacts/models/resnet18_42_original.pt.


## Step 3: unlearning 

### neggrad 

In [7]:
!PYTHONPATH=../../src python3 -m unlearning.main \
    dataset=imagenet \
    model=resnet18 \
    model=torchvision \
    unlearner=neggrad \
    dataset.save_path="imagenet_example_split" \
    model.model_name=resnet18 \
    model.model_ckpt_path="artifacts/models/resnet18_42_original.pt" \
    unlearner.cfg.lr=0.001 \
    unlearner.cfg.lr_decay_factor=0 \
    unlearner.cfg.epochs=1 \
    unlearner.evaluate=false \
    +wandb_cfg=default\
    ++wandb_cfg.run_id=neggrad_job_013


============ Run Configuration ============
seed: 42
verbose: true
output_dir: ./artifacts/models
id: null
model:
  load_method: torchvision
  model_name: resnet18
  model_ckpt_path: artifacts/models/resnet18_42_original.pt
dataset:
  name: imagenet
  num_classes: 1000
  save_path: imagenet_example_split
  cfg:
    batch_sizes:
      retain: 64
      forget: 64
      val: 64
    num_workers: 4
unlearner:
  name: neggrad
  evaluate: false
  cfg:
    optimizer: sgd
    lr: 0.001
    momentum: null
    lr_decay_factor: 0
    epochs_per_lr_decay: 2
    weight_decay: 0
    use_l2_penalty: false
    epochs: 1
wandb_cfg:
  project_name: machine_unlearning_001
  run_id: neggrad_job_013

Using device: cuda
wandb: Currently logged in as: aleksandra-pasieka32 (aleksandra-pasieka32-imperial-college-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Tracki

### scrub

In [8]:
!PYTHONPATH=../../src python3 -m unlearning.main \
    dataset=imagenet \
    model=resnet18 \
    model=torchvision \
    unlearner=scrub \
    dataset.save_path="imagenet_example_split" \
    model.model_name=resnet18 \
    model.model_ckpt_path="artifacts/models/resnet18_42_original.pt" \
    unlearner.cfg.lr=0.001 \
    unlearner.cfg.lr_decay_factor=0 \
    unlearner.cfg.min_epochs=0 \
    unlearner.cfg.momentum=0 \
    unlearner.cfg.weight_decay=0 \
    unlearner.cfg.max_epochs=1 \
    unlearner.cfg.alpha=1 \
    unlearner.cfg.gamma=0.5 \
    +wandb_cfg=default \
    ++wandb_cfg.run_id=scrub_job_014


============ Run Configuration ============
seed: 42
verbose: true
output_dir: ./artifacts/models
id: null
model:
  load_method: torchvision
  model_name: resnet18
  model_ckpt_path: artifacts/models/resnet18_42_original.pt
dataset:
  name: imagenet
  num_classes: 1000
  save_path: imagenet_example_split
  cfg:
    batch_sizes:
      retain: 64
      forget: 64
      val: 64
    num_workers: 4
unlearner:
  name: scrub
  evaluate: true
  cfg:
    optimizer: sgd
    lr: 0.001
    lr_decay_factor: 0
    epochs_per_lr_decay: 2
    momentum: 0
    weight_decay: 0
    use_l2_penalty: false
    min_epochs: 0
    max_epochs: 1
    alpha: 1
    gamma: 0.5
    sep_epochs: true
wandb_cfg:
  project_name: machine_unlearning_001
  run_id: scrub_job_014

Using device: cuda
wandb: Currently logged in as: aleksandra-pasieka32 (aleksandra-pasieka32-imperial-college-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to 

## Step 4: GGL Reconstruction

In [ ]:
!PYTHONPATH=../../src python3 -m attacks.main_reconstructor \
    data=imagenet \
    reconstructor=ggl \
    original_weights="artifacts/models/unlearn/neggrad/resnet18_42_original.pt" \
    unlearned_weights="artifacts/models/unlearn/neggrad/resnet18_42_unlearned.pt" \
    data.model_name=resnet18 \
    data.labels=[12] \
    data.data_root="imagenet_example_split/forget" \
    reconstructor.cfg.budget=1000 \
    reconstructor.lr=0.001 \
    +wandb_cfg=default \
    +wandb_cfg.extra_config.unlearning_method=neggrad \
    reconstructor.cfg.initial_lr=1 \
    +wandb_cfg.extra_config.epochs=1


============ Run Configuration ============
experiment_name: reconstruction_experiment
original_weights: artifacts/models/unlearn/neggrad/resnet18_42_original.pt
unlearned_weights: artifacts/models/unlearn/neggrad/resnet18_42_unlearned.pt
output_dir: ./artifacts/
seed: 42
verbose: true
data:
  model_name: resnet18
  dataset_name: imagenet
  labels:
  - 12
  num_classes: 1000
  data_root: imagenet_example_split/forget
reconstructor:
  type: ggl
  lr: 0.001
  cfg:
    num_updates: 1
    unlearning_method: neggrad
    batch_size: 1
    loss_models: l2
    budget: 1000
    search_dim: 128
    use_tanh: false
    gp_optim: AdamW
    use_scheduler: true
    initial_lr: 1
    initial_z_path: null
  unlearner:
    alpha: None
    gamma: None
    min_epochs: None
    max_epochs: None
    weight_decay: 0
    use_l2_penalty: false
    optimizer: sgd
    momentum: 0
    sep_epochs: true
    lr_decay_factor: null
    epochs_per_lr_decay: null
wandb_cfg:
  project: MU reconstruction
  extra_config:


In [ ]:
!PYTHONPATH=../../src python3 -m attacks.main_reconstructor \
    data=imagenet \
    reconstructor=ggl \
    original_weights="artifacts/models/unlearn/scrub/resnet18_42_original.pt" \
    unlearned_weights="artifacts/models/unlearn/scrub/resnet18_42_unlearned.pt" \
    data.model_name=resnet18 \
    data.labels=[12] \
    data.data_root="imagenet_example_split/forget" \
    reconstructor.cfg.budget=1000 \
    reconstructor.lr=0.001 \
    reconstructor.cfg.unlearning_method=scrub \
    reconstructor.unlearner.alpha=1 \
    reconstructor.unlearner.gamma=0.5 \
    reconstructor.unlearner.min_epochs=0 \
    reconstructor.unlearner.max_epochs=1 \
    reconstructor.cfg.initial_lr=1 \
    +wandb_cfg=default \
    +wandb_cfg.extra_config.unlearning_method=scrub \
    +wandb_cfg.extra_config.epochs=1


============ Run Configuration ============
experiment_name: reconstruction_experiment
original_weights: artifacts/models/unlearn/scrub/resnet18_42_original_004.pt
unlearned_weights: artifacts/models/unlearn/scrub/resnet18_42_unlearned_004.pt
output_dir: ./artifacts/
seed: 42
verbose: true
data:
  model_name: resnet18
  dataset_name: imagenet
  labels:
  - 12
  num_classes: 1000
  data_root: imagenet_example_split/forget
reconstructor:
  type: ggl
  lr: 0.001
  cfg:
    num_updates: 1
    unlearning_method: scrub
    batch_size: 1
    loss_models: l2
    budget: 1000
    search_dim: 128
    use_tanh: false
    gp_optim: AdamW
    use_scheduler: true
    initial_lr: 1
    initial_z_path: null
  unlearner:
    alpha: 1
    gamma: 0.5
    min_epochs: 0
    max_epochs: 1
    weight_decay: 0
    use_l2_penalty: false
    optimizer: sgd
    momentum: 0
    sep_epochs: true
    lr_decay_factor: null
    epochs_per_lr_decay: null
wandb_cfg:
  project: MU reconstruction
  extra_config:
    unle